# POPS: Priority-Order Portfolio Search for Cooperative A*

This notebook compares fixed-priority Cooperative A* against POPS on the same grid and robot start/goal positions.

**Files needed in the same folder:** `cooperative_astar.py`, `pops_cooperative_astar.py`, and a `results/` folder containing `grid.csv`, `start.csv`, and `goal.csv`.

In [ ]:
import os
import time
import importlib
import numpy as np
import matplotlib.pyplot as plt

import cooperative_astar
import pops_cooperative_astar
importlib.reload(cooperative_astar)
importlib.reload(pops_cooperative_astar)
from cooperative_astar import prioritized_cooperative_astar
from pops_cooperative_astar import prioritized_order_portfolio_search


## 1. Load the grid and robot positions

In [ ]:
RESULTS_DIR = "results"
grid_file = os.path.join(RESULTS_DIR, "grid.csv")
start_file = os.path.join(RESULTS_DIR, "start.csv")
goal_file = os.path.join(RESULTS_DIR, "goal.csv")

for filename in (grid_file, start_file, goal_file):
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"Missing {filename}. Run your grid generator first and save the CSV files."
        )

grid = np.loadtxt(grid_file, delimiter=",", dtype=int)
starts_array = np.atleast_2d(np.loadtxt(start_file, delimiter=",", dtype=int))
goals_array = np.atleast_2d(np.loadtxt(goal_file, delimiter=",", dtype=int))
starts = [tuple(map(int, row)) for row in starts_array]
goals = [tuple(map(int, row)) for row in goals_array]

print("Grid shape:", grid.shape)
print("Number of robots:", len(starts))
print("Starts:", starts)
print("Goals:", goals)


## 2. Run fixed-priority Cooperative A*

In [ ]:
start_time = time.perf_counter()
fixed_results = prioritized_cooperative_astar(grid, starts, goals)
fixed_runtime = time.perf_counter() - start_time
print(f"Fixed-priority runtime: {fixed_runtime:.6f} seconds")


## 3. Run POPS

In [ ]:
# For up to 4 robots, POPS evaluates every priority permutation.
# For more robots, it evaluates up to MAX_ORDERS deterministic/randomized orders.
MAX_ORDERS = 120
SEED = 42

start_time = time.perf_counter()
pops_output = prioritized_order_portfolio_search(
    grid, starts, goals, max_orders=MAX_ORDERS, seed=SEED
)
pops_runtime = time.perf_counter() - start_time
pops_results = pops_output["results"]

print("Selected priority order:", pops_output["best_priority_order"])
print("Orders tested:", pops_output["orders_tested"])
print("All robots successfully planned:", pops_output["all_success"])
print("Best tested makespan:", pops_output["best_makespan"])
print("Best tested total path length:", pops_output["best_total_path_length"])
print(f"POPS runtime: {pops_runtime:.6f} seconds")


## 4. Calculate comparable metrics

In [ ]:
def summarize(results, number_of_robots):
    robot_results = [results[f"Robot {i + 1}"] for i in range(number_of_robots)]
    success_count = sum(result["success"] for result in robot_results)
    success_rate = 100 * success_count / number_of_robots if number_of_robots else 0
    arrival_times = [result["arrival_time"] for result in robot_results if result["success"]]
    path_lengths = [result["path_length"] for result in robot_results if result["success"]]
    nodes_expanded = sum(result["nodes_expanded"] for result in robot_results)
    makespan = max(arrival_times) if arrival_times else None
    total_path_length = sum(path_lengths)
    return {
        "success_count": success_count,
        "success_rate_percent": success_rate,
        "makespan": makespan,
        "total_path_length": total_path_length,
        "nodes_expanded": nodes_expanded,
    }

fixed_metrics = summarize(fixed_results, len(starts))
pops_metrics = summarize(pops_results, len(starts))

print(f"{'Metric':<28}{'Fixed Cooperative A*':>22}{'POPS':>14}")
print("-" * 64)
for key, label in [
    ("success_count", "Successful robots"),
    ("success_rate_percent", "Success rate (%)"),
    ("makespan", "Makespan"),
    ("total_path_length", "Total path length"),
    ("nodes_expanded", "Nodes expanded"),
]:
    print(f"{label:<28}{str(fixed_metrics[key]):>22}{str(pops_metrics[key]):>14}")
print(f"{'Runtime (seconds)':<28}{fixed_runtime:>22.6f}{pops_runtime:>14.6f}")

if fixed_metrics["makespan"] is not None and pops_metrics["makespan"] is not None and fixed_metrics["makespan"] > 0:
    reduction = 100 * (fixed_metrics["makespan"] - pops_metrics["makespan"]) / fixed_metrics["makespan"]
    print(f"POPS makespan reduction: {reduction:.2f}% (negative means POPS was worse)")


## 5. Inspect the tested priority orders

In [ ]:
sorted_trials = sorted(
    pops_output["trials"],
    key=lambda trial: (
        not trial["all_success"],
        trial["makespan"],
        trial["total_path_length"],
    ),
)
for trial in sorted_trials[:10]:
    print(
        f"Order={trial['priority_order']}, all_success={trial['all_success']}, "
        f"makespan={trial['makespan']}, total_path_length={trial['total_path_length']}"
    )


## 6. Visualize both schedules

In [ ]:
def visualize_paths(grid, starts, goals, results, title):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=1)
    colors = plt.cm.tab20(np.linspace(0, 1, max(1, len(starts))))
    for i, (start, goal) in enumerate(zip(starts, goals)):
        result = results[f"Robot {i + 1}"]
        color = colors[i]
        if result["success"]:
            path = result["path"]
            ax.plot([p[1] for p in path], [p[0] for p in path], color=color,
                    linewidth=2, marker=".", markersize=4, label=f"R{i + 1} path")
        else:
            print(f"Robot {i + 1} has no path in {title}.")
        ax.scatter(start[1], start[0], color=color, marker="o", s=130,
                   edgecolors="black", label=f"R{i + 1} start")
        ax.scatter(goal[1], goal[0], color=color, marker="*", s=180,
                   edgecolors="black", label=f"R{i + 1} goal")
    ax.set_xticks(np.arange(-0.5, cols, 1))
    ax.set_yticks(np.arange(-0.5, rows, 1))
    ax.grid(color="gray", linewidth=0.5)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(rows - 0.5, -0.5)
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

visualize_paths(grid, starts, goals, fixed_results, "Fixed-Priority Cooperative A*")
visualize_paths(grid, starts, goals, pops_results, "POPS: Best Tested Priority Order")


## 7. Interpretation

- POPS selects the best result among the priority orders it tested.
- It does **not** guarantee globally optimal MAPF makespan.
- Runtime should generally be higher because multiple prioritized searches are run.
- Use the values printed above as measured results for this exact input instance; repeat with other saved grids before drawing broad conclusions.